# Module 4: Function Approximation
## Notebook 1: Tile Coding

**Learning Objectives**

By completing this notebook, you will:
1. Implement tile coding from scratch and understand how it discretizes continuous state spaces
2. Apply tile coding to Mountain Car (Gymnasium) as a practical feature extractor
3. Compare tile coding against naive uniform discretization on coverage and generalization
4. Visualize how different tilings overlap to provide smooth generalization

**Prerequisites**
- Tabular Q-learning (Module 3)
- Linear function approximation concept (value as dot product of weights and features)
- Basic NumPy

**Estimated Time: 14 minutes**

---

### Why Function Approximation?

Tabular methods store one value per state. Mountain Car has a **continuous** 2D state space (position × velocity), making a lookup table impossible. We need to **generalize**: learning that state $(x, v)$ has value $-10$ should tell us something about the neighboring state $(x + 0.01, v)$.

**Tile coding** is the classical solution: partition the continuous space into overlapping grids ("tilings"), and represent each state as a binary feature vector indicating which tiles it falls in.

In [ ]:
# Purpose: Import dependencies and set seeds
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import gymnasium as gym

np.random.seed(42)

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("Setup complete.")

# Confirm gymnasium is available
env = gym.make('MountainCar-v0')
obs, _ = env.reset(seed=42)
print(f"MountainCar observation: {obs}  (position, velocity)")
print(f"Position range: [{env.observation_space.low[0]:.2f}, {env.observation_space.high[0]:.2f}]")
print(f"Velocity range: [{env.observation_space.low[1]:.4f}, {env.observation_space.high[1]:.4f}]")
env.close()

## 1. Naive Uniform Discretization

Before tile coding, let's understand the baseline: divide each dimension into $n$ equal bins. State $(x, v)$ maps to the bin pair $(i, j)$, giving $n^2$ discrete states. Problems with this approach:
- No generalization between adjacent bins
- Each state learned independently — poor sample efficiency
- Must choose bin size carefully (too coarse = bad approximation, too fine = too many states)

In [ ]:
# Purpose: Implement naive uniform discretization for comparison
# Key Concept: Uniform grids have sharp discontinuities at bin boundaries

class UniformDiscretizer:
    """
    Divides each dimension into n_bins equal intervals.
    Maps continuous state to a flat integer index.
    """

    def __init__(self, low, high, n_bins):
        """
        Parameters
        ----------
        low    : array-like, shape (n_dims,) — lower bounds per dimension
        high   : array-like, shape (n_dims,) — upper bounds per dimension
        n_bins : int — number of bins per dimension
        """
        self.low    = np.array(low, dtype=float)
        self.high   = np.array(high, dtype=float)
        self.n_bins = n_bins
        self.n_dims = len(low)
        self.n_states = n_bins ** self.n_dims

    def encode(self, state):
        """
        Map continuous state to integer index.

        Returns
        -------
        int in [0, n_states)
        """
        state = np.clip(state, self.low, self.high)
        # Normalize to [0, 1) then bin
        normalized = (state - self.low) / (self.high - self.low)
        bins = np.minimum((normalized * self.n_bins).astype(int), self.n_bins - 1)
        # Convert multi-dim bin to flat index
        idx = 0
        for b in bins:
            idx = idx * self.n_bins + b
        return int(idx)


# Mountain Car bounds
LOW  = np.array([-1.2,  -0.07])
HIGH = np.array([ 0.6,   0.07])

disc = UniformDiscretizer(LOW, HIGH, n_bins=20)
print(f"Uniform discretizer: {disc.n_states} states (20x20 grid)")

# Test encoding
test_states = [np.array([-1.2, 0.0]), np.array([0.0, 0.0]), np.array([0.6, 0.07])]
for s in test_states:
    print(f"  state={s} -> index={disc.encode(s)}")

## 2. Tile Coding: The Core Idea

Tile coding uses **multiple overlapping grids** (tilings). Each tiling is offset slightly from the others. A state activates exactly one tile per tiling, so it activates $n\_tilings$ tiles total.

```
Tiling 1:  |  A  |  B  |  C  |  D  |
Tiling 2:   | E  |  F  |  G  |  H  |
Tiling 3:    | I |  J  |  K  |  L  |
State x:          ^
  Active tiles: B, E, I   (one from each tiling)
```

Two nearby states share some active tiles → smooth generalization. States further apart share fewer tiles → less interference.

In [ ]:
# Purpose: Implement tile coding from scratch
# Key Concept: Multiple offset tilings create overlapping receptive fields

class TileCoder:
    """
    Multi-dimensional tile coding.

    Each of the n_tilings grids is offset by (tiling_idx / n_tilings) of a tile width
    in each dimension. A state maps to a binary feature vector with exactly
    n_tilings ones out of (n_tilings * n_tiles^n_dims) total features.
    """

    def __init__(self, low, high, n_tiles, n_tilings):
        """
        Parameters
        ----------
        low       : array-like, shape (n_dims,)
        high      : array-like, shape (n_dims,)
        n_tiles   : int — number of tiles per dimension per tiling
        n_tilings : int — number of overlapping tilings
        """
        self.low       = np.array(low, dtype=float)
        self.high      = np.array(high, dtype=float)
        self.n_tiles   = n_tiles
        self.n_tilings = n_tilings
        self.n_dims    = len(low)

        # Width of each tile in each dimension
        self.tile_width = (self.high - self.low) / n_tiles

        # Total features: one binary feature per tile per tiling
        self.n_features = n_tilings * (n_tiles ** self.n_dims)

    def get_active_tiles(self, state):
        """
        Return the indices of active tiles for a given state.

        Parameters
        ----------
        state : array-like, shape (n_dims,)

        Returns
        -------
        active_tiles : list of int, length n_tilings
            Each entry is the flat index of the active tile in that tiling.
        """
        state = np.clip(state, self.low, self.high - 1e-8)
        active_tiles = []

        tiles_per_tiling = self.n_tiles ** self.n_dims

        for tiling_idx in range(self.n_tilings):
            # Offset: each tiling is shifted by (tiling_idx / n_tilings) * tile_width
            offset  = (tiling_idx / self.n_tilings) * self.tile_width
            shifted = state - self.low + offset

            # Which tile does the shifted state fall into?
            tile_coords = np.floor(shifted / self.tile_width).astype(int)
            tile_coords = np.clip(tile_coords, 0, self.n_tiles - 1)

            # Convert n-d tile coordinate to flat index within this tiling
            flat_idx = 0
            for coord in tile_coords:
                flat_idx = flat_idx * self.n_tiles + coord

            # Global feature index: tiling offset + local tile index
            global_idx = tiling_idx * tiles_per_tiling + flat_idx
            active_tiles.append(global_idx)

        return active_tiles

    def encode(self, state):
        """
        Return binary feature vector for a state.

        Returns
        -------
        phi : np.ndarray, shape (n_features,), dtype float
            Exactly n_tilings ones, rest zeros.
        """
        phi = np.zeros(self.n_features)
        for tile_idx in self.get_active_tiles(state):
            phi[tile_idx] = 1.0
        return phi


# Create a tile coder for Mountain Car
tc = TileCoder(low=LOW, high=HIGH, n_tiles=8, n_tilings=8)
print(f"TileCoder: {tc.n_features} features ({tc.n_tilings} tilings x {tc.n_tiles}^2 tiles)")

# Test encoding
s = np.array([-0.6, 0.0])   # typical starting position
active = tc.get_active_tiles(s)
phi    = tc.encode(s)
print(f"State {s}: active tiles = {active}")
print(f"Feature vector: {int(phi.sum())} ones out of {len(phi)} features")

## 3. Visualizing Tile Coverage

Let's draw the first two tilings over the Mountain Car position-velocity space to see how the offsets create overlap.

In [ ]:
# Purpose: Visualize tiling grids overlaid on the state space
# Key Concept: Offset tilings create gradual transitions between discrete regions

def plot_tilings(tc, n_show=3):
    fig, ax = plt.subplots(figsize=(11, 6))

    colors = plt.cm.Set1(np.linspace(0, 0.8, n_show))
    pos_range = tc.high[0] - tc.low[0]
    vel_range = tc.high[1] - tc.low[1]

    for t_idx in range(n_show):
        offset = (t_idx / tc.n_tilings) * tc.tile_width
        color  = colors[t_idx]

        # Draw tile boundaries for this tiling
        for i in range(tc.n_tiles + 1):
            x = tc.low[0] - offset[0] + i * tc.tile_width[0]
            ax.axvline(x, color=color, alpha=0.6, linewidth=1.2,
                       label=f'Tiling {t_idx+1}' if i == 0 else None)
        for j in range(tc.n_tiles + 1):
            y = tc.low[1] - offset[1] + j * tc.tile_width[1]
            ax.axhline(y, color=color, alpha=0.6, linewidth=1.2)

    # Highlight a sample state
    s = np.array([-0.6, 0.02])
    ax.scatter(*s, s=200, color='black', zorder=10, label='Sample state')

    # Shade the tiles activated by this state
    for t_idx in range(n_show):
        offset     = (t_idx / tc.n_tilings) * tc.tile_width
        shifted    = s - tc.low + offset
        tile_c     = np.floor(shifted / tc.tile_width).astype(int)
        tile_c     = np.clip(tile_c, 0, tc.n_tiles - 1)
        tile_x0    = tc.low[0] - offset[0] + tile_c[0] * tc.tile_width[0]
        tile_y0    = tc.low[1] - offset[1] + tile_c[1] * tc.tile_width[1]
        rect = patches.Rectangle(
            (tile_x0, tile_y0),
            tc.tile_width[0], tc.tile_width[1],
            linewidth=2, edgecolor=colors[t_idx],
            facecolor=colors[t_idx], alpha=0.25
        )
        ax.add_patch(rect)

    ax.set_xlim(tc.low[0] - 0.05, tc.high[0] + 0.05)
    ax.set_ylim(tc.low[1] - 0.005, tc.high[1] + 0.005)
    ax.set_xlabel('Position', fontsize=12)
    ax.set_ylabel('Velocity', fontsize=12)
    ax.set_title(f'Tile Coding: First {n_show} of {tc.n_tilings} Tilings\n'
                 f'(Shaded = tiles activated by black dot)', fontsize=13)
    ax.legend(fontsize=10, loc='upper left')
    plt.tight_layout()
    plt.savefig('../resources/tile_coding_visualization.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_tilings(tc, n_show=3)

## 4. Measuring Generalization: Tile Overlap Between States

A key property of tile coding is that nearby states share more active tiles (higher overlap) than distant states. Let's measure this directly.

In [ ]:
# Purpose: Quantify generalization by measuring tile overlap between states
# Key Concept: Overlap = number of shared active tiles / n_tilings

def tile_overlap(tc, s1, s2):
    """
    Fraction of tilings in which s1 and s2 activate the same tile.
    Ranges from 0 (no shared tiles) to 1 (identical tile activation).
    """
    tiles1 = set(tc.get_active_tiles(s1))
    tiles2 = set(tc.get_active_tiles(s2))
    return len(tiles1 & tiles2) / tc.n_tilings


# Reference state: center of position range
s_ref = np.array([-0.6, 0.0])

# Vary position while keeping velocity fixed
positions  = np.linspace(LOW[0], HIGH[0], 200)
overlaps_tc   = [tile_overlap(tc, s_ref, np.array([p, 0.0])) for p in positions]

# Compare with uniform discretizer: overlap = 1 if same bin, 0 otherwise
ref_bin = disc.encode(s_ref)
overlaps_disc = [1.0 if disc.encode(np.array([p, 0.0])) == ref_bin else 0.0
                 for p in positions]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(positions, overlaps_tc,   color='steelblue', linewidth=2,
        label=f'Tile Coding ({tc.n_tilings} tilings, {tc.n_tiles} tiles/dim)')
ax.plot(positions, overlaps_disc, color='tomato',    linewidth=2,
        label='Uniform Discretization (20 bins)')
ax.axvline(s_ref[0], color='black', linestyle='--', linewidth=1, label='Reference state')
ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('Generalization Overlap', fontsize=12)
ax.set_title('Generalization: Tile Coding vs. Uniform Discretization', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../resources/generalization_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

### Exercise 4.1.1: Number of Tilings vs. Feature Count

**Task:** There is a tension in tile coding: more tilings give smoother generalization but increase the feature vector size and memory. Compute the feature count for tile configurations with n_tilings in [1, 2, 4, 8, 16] and n_tiles=8. Plot feature count vs. n_tilings. Then compute the tile overlap between states 0.1 apart in position for each configuration.

**Expected Output:** Two subplots — one showing feature count growth, one showing overlap vs. distance for each n_tilings.

<details>
<summary>Hint 1</summary>
Use tc.n_features to get the total feature count. For each n_tilings configuration, create a new TileCoder and measure tile_overlap(tc, s_ref, s_ref + [0.1, 0]).
</details>

<details>
<summary>Hint 2</summary>
With n_tilings=1, generalization is binary (same as uniform). With n_tilings=8, there are 7 possible overlap values (0, 1/8, 2/8, ..., 1), giving smooth transitions.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

n_tilings_list = [1, 2, 4, 8, 16]
feature_counts = []
overlap_at_01  = []   # overlap between states 0.1 apart in position

s_a = np.array([-0.6, 0.0])
s_b = np.array([-0.5, 0.0])   # 0.1 apart

for n_t in n_tilings_list:
    tc_test = TileCoder(LOW, HIGH, n_tiles=8, n_tilings=n_t)
    feature_counts.append(tc_test.n_features)
    overlap_at_01.append(tile_overlap(tc_test, s_a, s_b))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar([str(n) for n in n_tilings_list], feature_counts,
            color='steelblue', edgecolor='navy')
axes[0].set_xlabel('Number of Tilings', fontsize=12)
axes[0].set_ylabel('Total Feature Count', fontsize=12)
axes[0].set_title('Feature Count vs. Number of Tilings (8 tiles/dim)', fontsize=12)

axes[1].plot([str(n) for n in n_tilings_list], overlap_at_01,
             'o-', color='tomato', linewidth=2, markersize=9)
axes[1].set_xlabel('Number of Tilings', fontsize=12)
axes[1].set_ylabel('Overlap (states 0.1 apart)', fontsize=12)
axes[1].set_title('Generalization Overlap vs. Number of Tilings', fontsize=12)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

for n_t, fc, ov in zip(n_tilings_list, feature_counts, overlap_at_01):
    print(f"n_tilings={n_t:2d}: features={fc:5d}, overlap_0.1={ov:.3f}")

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_411():
    assert len(feature_counts) == len(n_tilings_list), \
        "Compute feature_counts for all n_tilings_list values"
    # Feature count grows linearly with n_tilings
    assert feature_counts[-1] > feature_counts[0], \
        "More tilings should mean more features"
    assert feature_counts[1] == 2 * feature_counts[0], \
        "Feature count should double when n_tilings doubles"
    # With 1 tiling, overlap is binary (0 or 1)
    assert overlap_at_01[0] in [0.0, 1.0], \
        f"With 1 tiling, overlap should be 0 or 1, got {overlap_at_01[0]}"
    print("Exercise 4.1.1 passed!")

test_exercise_411()

### Exercise 4.1.2: Action-Value Tile Coding

**Task:** For TD control with function approximation, we need to encode state-action pairs, not just states. Implement an `encode_state_action(state, action, n_actions)` method: create separate feature blocks for each action so that features for different actions never overlap.

The feature vector should have length `n_actions * n_features`, with the state features placed in the block corresponding to the action index, and zeros elsewhere.

**Expected Output:** For a 3-action problem, encode_state_action returns a vector 3x as long as encode_state, with non-zeros only in the action's block.

<details>
<summary>Hint 1</summary>
Create a zero vector of length n_actions * tc.n_features. Then set the slice [action * tc.n_features : (action+1) * tc.n_features] using tc.encode(state).
</details>

<details>
<summary>Hint 2</summary>
This is called "separate tilings per action" and is the standard approach for approximate Q-learning.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def encode_state_action(tc, state, action, n_actions):
    """
    Create a feature vector for a (state, action) pair using separate feature
    blocks per action.

    Parameters
    ----------
    tc       : TileCoder
    state    : np.ndarray, shape (n_dims,)
    action   : int in [0, n_actions)
    n_actions: int

    Returns
    -------
    phi : np.ndarray, shape (n_actions * tc.n_features,)
        Exactly n_tilings ones in the block for 'action', zeros elsewhere.
    """
    phi = np.zeros(n_actions * tc.n_features)
    state_phi = tc.encode(state)
    start = action * tc.n_features
    phi[start : start + tc.n_features] = state_phi
    return phi


# Test with Mountain Car (3 actions: push left, nothing, push right)
N_ACTIONS = 3
s = np.array([-0.6, 0.0])

for a in range(N_ACTIONS):
    phi = encode_state_action(tc, s, a, N_ACTIONS)
    nonzero_range = (phi.nonzero()[0].min(), phi.nonzero()[0].max())
    print(f"Action {a}: phi length={len(phi)}, nonzero indices [{nonzero_range[0]}, {nonzero_range[1]}], "
          f"sum={phi.sum():.0f}")

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_412():
    N_ACTIONS = 3
    s = np.array([-0.6, 0.0])

    for a in range(N_ACTIONS):
        phi = encode_state_action(tc, s, a, N_ACTIONS)
        assert len(phi) == N_ACTIONS * tc.n_features, \
            f"phi length should be {N_ACTIONS * tc.n_features}, got {len(phi)}"
        assert phi.sum() == tc.n_tilings, \
            f"Action {a}: expected {tc.n_tilings} ones, got {phi.sum():.0f}"

        # Check that non-zeros are in the correct block
        nonzero_idx = phi.nonzero()[0]
        assert all(a * tc.n_features <= i < (a+1) * tc.n_features for i in nonzero_idx), \
            f"Action {a}: nonzeros should be in block [{a*tc.n_features}, {(a+1)*tc.n_features})"

    # Different actions should have non-overlapping non-zeros
    phi0 = encode_state_action(tc, s, 0, N_ACTIONS)
    phi1 = encode_state_action(tc, s, 1, N_ACTIONS)
    assert np.dot(phi0, phi1) == 0, "Features for different actions should not overlap"

    print("Exercise 4.1.2 passed!")

test_exercise_412()

## Key Takeaways

1. **Naive discretization has no generalization**: each bin is independent, so the agent must visit every bin to learn its value. Tile coding shares information between nearby states.
2. **Multiple offset tilings create smooth generalization**: two states 0.05 apart might share 6 of 8 tiles (75% overlap); 0.3 apart might share 0. The gradient is smooth, not binary.
3. **Feature vectors are sparse**: only `n_tilings` of `n_tilings * n_tiles^d` features are active. This enables efficient dot-product computation for linear function approximation.
4. **Action-conditional features**: placing separate feature blocks per action lets us represent Q(s, a) = w · phi(s, a) with a single weight vector.
5. **The n_tilings / n_tiles trade-off**: more tilings = smoother generalization and better precision, but more parameters. A rule of thumb: n_tilings = 8 works well for 2D continuous spaces.

## What's Next

Notebook 2 uses tile coding features as inputs to **semi-gradient TD(0)** with a linear weight vector. We approximate the value function as $V(s) \approx \mathbf{w}^\top \phi(s)$ and derive the gradient-based update rule. We also explore polynomial and Fourier basis functions as alternative feature representations.